In [3]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from datetime import datetime

In [8]:
def preprocess(df_input, reference_date=None, is_train=True):
    df = df_input.copy()
    
    if reference_date is None:
        reference_date = pd.Timestamp.today()

    # ── 1. Drop ID / irrelevant columns ──────────────────────────────────
    drop_always = [
        'quote_id', 'vehicle_number_plate',
        'second_driver_birthdate',  # 100% null
        'second_driver_claim_free_years',  # 100% null
        'postal_code_houses_owned_by_rental_association_ratio',  # 100% null
    ]
    df.drop(columns=[c for c in drop_always if c in df.columns], inplace=True)

    # ── 2. Drop zero variance / useless ──────────────────────────────────
    drop_low_variance = [
        'usage', 'vehicle_is_taxi', 'vehicle_number_of_wheels',
        'is_driver_owner', 'vehicle_can_be_registered',
        'vehicle_is_marked_for_export', 'vehicle_model',
        'municipality', 'postal_code',
        'postal_code_houses_built_before_1945_ratio',
        'postal_code_houses_built_45_to_65_ratio',
        'postal_code_houses_built_65_to_75_ratio',
        'postal_code_houses_built_75_to_85_ratio',
        'postal_code_houses_built_85_to_95_ratio',
        'postal_code_houses_built_95_to_05_ratio',
        'postal_code_houses_built_05_to_15_ratio',
    ]
    df.drop(columns=[c for c in drop_low_variance if c in df.columns], inplace=True)

    # ── 3. Parse dates → numeric features ────────────────────────────────
    # Flag inspection before dropping
    if 'vehicle_inspection_report_date' in df.columns:
        df['has_inspection'] = df['vehicle_inspection_report_date'].notnull().astype(int)

    date_cols = [
        'vehicle_first_registration_date',
        'vehicle_country_first_registration_date',
        'vehicle_last_registration_date',
        'vehicle_inspection_report_date',
        'vehicle_inspection_expiry_date',
    ]
    for col in [c for c in date_cols if c in df.columns]:
        df[col] = pd.to_datetime(df[col], dayfirst=True, errors='coerce')

    if 'vehicle_first_registration_date' in df.columns and \
       'vehicle_years_since_first_registration' not in df.columns:
        df['vehicle_years_since_first_registration'] = (
            (reference_date - df['vehicle_first_registration_date']).dt.days // 365
        )
    if 'vehicle_inspection_expiry_date' in df.columns:
        df['vehicle_days_until_inspection_expiry'] = (
            df['vehicle_inspection_expiry_date'] - reference_date
        ).dt.days.clip(lower=-3650)
    if 'vehicle_last_registration_date' in df.columns and \
       'vehicle_years_since_last_registration' not in df.columns:
        df['vehicle_years_since_last_registration'] = (
            (reference_date - df['vehicle_last_registration_date']).dt.days // 365
        )

    df.drop(columns=[c for c in date_cols if c in df.columns], inplace=True)

    # ── 4. Birthdate → age ───────────────────────────────────────────────
    if 'contractor_birthdate' in df.columns:
        df['contractor_birthdate'] = pd.to_datetime(
            df['contractor_birthdate'], dayfirst=True, errors='coerce'
        )
        df['contractor_age'] = (
            (reference_date - df['contractor_birthdate']).dt.days // 365
        )
        df.drop(columns=['contractor_birthdate'], inplace=True)

    # ── 5. Convert numeric strings to float ──────────────────────────────
    to_numeric_cols = [
        'vehicle_engine_size', 'vehicle_power', 'vehicle_net_weight',
        'vehicle_gross_weight', 'vehicle_length', 'vehicle_width',
        'vehicle_height', 'vehicle_net_max_power', 'vehicle_net_max_power_electric',
        'vehicle_nominal_continuous_max_power', 'vehicle_power_to_net_weight_ratio',
        'vehicle_number_of_cylinders', 'vehicle_number_of_doors', 'vehicle_number_of_seats',
        'vehicle_value_new', 'vehicle_planned_annual_mileage', 'vehicle_age',
        'vehicle_years_since_country_first_registration',
        'vehicle_year_of_last_odometer_report',
        'postal_code_latitude', 'postal_code_longitude', 'postal_code_distance_to_border',
        'postal_code_population', 'postal_code_households', 'postal_code_houses',
        'postal_code_average_household_size', 'postal_code_average_property_value',
        'postal_code_address_density', 'municipality_crimes_per_1000',
        'postal_code_0_to_15_years_inhabitants_ratio',
        'postal_code_25_to_45_years_inhabitants_ratio',
        'postal_code_45_to_65_years_inhabitants_ratio',
        'postal_code_65_years_and_older_inhabitants_ratio',
        'postal_code_single_person_households_ratio',
        'postal_code_multi_person_households_without_children_ratio',
        'postal_code_two_parent_households_ratio',
        'postal_code_social_benefit_recipients_ratio',
        'postal_code_owner_occupied_houses_ratio',
        'postal_code_rental_houses_ratio',
        'postal_code_multi_family_houses_ratio',
        'postal_code_nearest_train_station_distance',
        'postal_code_nearest_train_transfer_station_distance',
        'postal_code_nearest_motorway_ramp_distance',
        'postal_code_nearest_hospital_excl_clinic_distance',
        'postal_code_nearest_hospital_incl_clinic_distance',
        'postal_code_nearest_general_practitioner_distance',
        'postal_code_nearest_gp_post_service_distance',
        'postal_code_nearest_pharmacy_distance',
        'postal_code_nearest_primary_school_distance',
        'postal_code_nearest_secondary_school_distance',
        'postal_code_nearest_secondary_school_havo_vwo_distance',
        'postal_code_nearest_secondary_school_vmbo_distance',
        'postal_code_nearest_supermarket_distance',
        'postal_code_nearest_groceries_distance',
        'postal_code_nearest_restaurant_distance',
        'postal_code_nearest_cafe_distance',
        'postal_code_nearest_snackbar_distance',
        'postal_code_nearest_hotel_distance',
        'postal_code_nearest_library_distance',
        'postal_code_nearest_museum_distance',
        'postal_code_nearest_cinema_distance',
        'postal_code_nearest_department_store_distance',
        'postal_code_nearest_fire_station_distance',
        'postal_code_nearest_daycare_distance',
        'postal_code_nearest_after_school_care_distance',
        'postal_code_nearest_attraction_distance',
        'postal_code_nearest_performance_venue_distance',
        'postal_code_nearest_music_venue_distance',
        'postal_code_nearest_swimming_pool_distance',
        'postal_code_nearest_ice_rink_distance',
        'postal_code_nearest_sauna_distance',
        'postal_code_nearest_tanning_salon_distance',
        'postal_code_hospitals_excl_clinic_within_10_km',
        'postal_code_hospitals_incl_clinic_within_10_km',
        'postal_code_general_practitioners_within_3_km',
        'postal_code_primary_schools_within_3_km',
        'postal_code_secondary_schools_within_5_km',
        'postal_code_secondary_schools_havo_vwo_within_5_km',
        'postal_code_secondary_schools_vmbo_within_5_km',
        'postal_code_supermarkets_within_3_km',
        'postal_code_groceries_within_3_km',
        'postal_code_restaurants_within_3_km',
        'postal_code_cafes_within_3_km',
        'postal_code_snackbars_within_3_km',
        'postal_code_hotels_within_10_km',
        'postal_code_museums_within_10_km',
        'postal_code_cinemas_within_10_km',
        'postal_code_department_stores_within_10_km',
        'postal_code_daycares_within_3_km',
        'postal_code_after_school_cares_within_3_km',
        'postal_code_attractions_within_20_km',
        'postal_code_performance_venues_within_10_km',
        'vehicle_ownership_duration', 'claim_free_years',
        'postal_code_urban_category',
    ]
    for col in [c for c in to_numeric_cols if c in df.columns]:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    # ── 6. Electric vehicle features ─────────────────────────────────────
    if 'vehicle_net_max_power_electric' in df.columns:
        df = df.copy()
        df['vehicle_is_electric'] = df['vehicle_net_max_power_electric'].notnull().astype(int)
        df['vehicle_net_max_power_electric'] = df['vehicle_net_max_power_electric'].fillna(0)
    if 'vehicle_nominal_continuous_max_power' in df.columns:
        df['vehicle_nominal_continuous_max_power'] = \
            df['vehicle_nominal_continuous_max_power'].fillna(0)

    # ── 7. Inspection status ──────────────────────────────────────────────
    if 'vehicle_inspection_number_of_deficiencies_found' in df.columns:
        df = df.copy()
        df['inspection_status'] = 0
        df.loc[df['vehicle_inspection_number_of_deficiencies_found'] == 1.0,
               'inspection_status'] = 1
        df.drop(columns=['vehicle_inspection_number_of_deficiencies_found',
                         'has_inspection'], inplace=True,
                errors='ignore')

    # ── 8. Binary mappings ────────────────────────────────────────────────
    binary_map = {'yes': 1, 'no': 0}
    binary_cols = [
        'vehicle_is_imported', 'vehicle_is_imported_within_last_12_months',
        'vehicle_has_open_recall',
    ]
    for col in [c for c in binary_cols if c in df.columns]:
        if df[col].dtype == object:
            df[col] = df[col].map(binary_map)
    df['vehicle_is_imported_within_last_12_months'] = \
        df['vehicle_is_imported_within_last_12_months'].fillna(0).astype(int)

    # ── 9. Payment frequency ──────────────────────────────────────────────
    if 'payment_frequency' in df.columns and df['payment_frequency'].dtype == object:
        df['payment_frequency'] = df['payment_frequency'].map(
            {'monthly': 0, 'yearly': 1}
        )

    df = df.copy()  # defragment
    return df



In [11]:

reference_date = pd.Timestamp('2026-01-01')  # fix date so both sets are consistent

df_raw = pd.read_parquet('../Hackathon2026Task-20260328T120731Z-3-001/Hackathon2026Task/block1_train.parquet')  # your actual filename
df_test_raw = pd.read_parquet('../Hackathon2026Task-20260328T120731Z-3-001/Hackathon2026Task/block2_test.parquet')  # your actual filename


df_train = preprocess(df_raw, reference_date=reference_date, is_train=True)
df_test  = preprocess(df_test_raw, reference_date=reference_date, is_train=False)

print(f"Train shape: {df_train.shape}")
print(f"Test shape:  {df_test.shape}")
print(f"\nTrain object cols: {df_train.select_dtypes(include='object').columns.tolist()}")
print(f"Test object cols:  {df_test.select_dtypes(include='object').columns.tolist()}")

Train shape: (541292, 133)
Test shape:  (164092, 122)

Train object cols: ['coverage', 'vehicle_maker', 'vehicle_fuel_type', 'vehicle_primary_color', 'vehicle_odometer_verdict_code', 'province']
Test object cols:  ['coverage', 'vehicle_maker', 'vehicle_fuel_type', 'vehicle_primary_color', 'vehicle_odometer_verdict_code', 'province']


In [12]:
train_not_test = set(df_train.columns) - set(df_test.columns)
test_not_train = set(df_test.columns) - set(df_train.columns)

print(f"In train but not test ({len(train_not_test)}):")
print(sorted(train_not_test))
print(f"\nIn test but not train ({len(test_not_train)}):")
print(sorted(test_not_train))

In train but not test (11):
['Insurer_A_price', 'Insurer_B_price', 'Insurer_C_price', 'Insurer_D_price', 'Insurer_E_price', 'Insurer_F_price', 'Insurer_G_price', 'Insurer_H_price', 'Insurer_I_price', 'Insurer_J_price', 'Insurer_K_price']

In test but not train (0):
[]


In [13]:
# Check correlation within postal code distance columns
dist_cols = [c for c in df_train.columns if 'nearest' in c]
size_cols = ['vehicle_length', 'vehicle_width', 'vehicle_height', 
             'vehicle_gross_weight', 'vehicle_net_weight']
ratio_cols = [c for c in df_train.columns if 'ratio' in c and 'postal' in c]
within_cols = [c for c in df_train.columns if 'within' in c]

print(f"Distance columns:    {len(dist_cols)}")
print(f"Vehicle size cols:   {len(size_cols)}")
print(f"Ratio columns:       {len(ratio_cols)}")
print(f"Within-radius cols:  {len(within_cols)}")
print(f"\nTotal reducible:     {len(dist_cols)+len(size_cols)+len(ratio_cols)+len(within_cols)}")
print(f"Current total cols:  {df_train.shape[1]}")

Distance columns:    32
Vehicle size cols:   5
Ratio columns:       11
Within-radius cols:  21

Total reducible:     69
Current total cols:  133


In [14]:
# Distance cols — keep only 3 most insurance-relevant
keep_distance = [
    'postal_code_nearest_motorway_ramp_distance',
    'postal_code_nearest_hospital_excl_clinic_distance',
    'postal_code_nearest_fire_station_distance',
]
drop_distance = [c for c in dist_cols if c not in keep_distance]

# Within-radius cols — keep only 3
keep_within = [
    'postal_code_hospitals_excl_clinic_within_10_km',
    'postal_code_supermarkets_within_3_km',
    'postal_code_restaurants_within_3_km',
]
drop_within = [c for c in within_cols if c not in keep_within]

# Vehicle size — keep only gross weight
drop_size = [c for c in size_cols if c != 'vehicle_gross_weight']

# Ratio cols — keep only 3
keep_ratio = [
    'postal_code_social_benefit_recipients_ratio',
    'postal_code_65_years_and_older_inhabitants_ratio',
    'postal_code_single_person_households_ratio',
]
drop_ratio = [c for c in ratio_cols if c not in keep_ratio]

# Other redundant cols
drop_other = [
    'vehicle_net_max_power_electric',
    'vehicle_nominal_continuous_max_power',
    'vehicle_years_since_country_first_registration',
    'vehicle_year_of_last_odometer_report',
    'postal_code_households',
    'postal_code_houses',
    'postal_code_average_household_size',
    'postal_code_latitude',
    'postal_code_longitude',
    'vehicle_days_until_inspection_expiry',  # 63% null, inspection_status covers this
    'vehicle_years_since_last_registration', # redundant with vehicle_age
]

# Combine all drops
all_drops = drop_distance + drop_within + drop_size + drop_ratio + drop_other
all_drops = [c for c in all_drops if c in df_train.columns]

print(f"Dropping {len(all_drops)} columns")
print(f"Remaining: {df_train.shape[1] - len(all_drops)} columns")

# Apply to both train and test
df_train.drop(columns=all_drops, inplace=True)
df_test.drop(columns=all_drops, inplace=True)

print(f"\nTrain shape: {df_train.shape}")
print(f"Test shape:  {df_test.shape}")

Dropping 70 columns
Remaining: 63 columns

Train shape: (541292, 63)
Test shape:  (164092, 52)


In [ ]:
# # ── One-hot encode low cardinality cols ──────────────────────────────────
# onehot_cols = [
#     'vehicle_fuel_type',           # 7 values
#     'vehicle_primary_color',       # 15 values
#     'vehicle_odometer_verdict_code', # 8 values
#     'province',                    # 12 values
# ]

# # Combine train and test before encoding to ensure same columns appear in both
# df_train['_is_train'] = 1
# df_test['_is_train'] = 0
# combined = pd.concat([df_train, df_test], axis=0)

# combined = pd.get_dummies(combined, columns=onehot_cols, drop_first=False)

# # ── Target encode high cardinality cols ──────────────────────────────────
# # Compute mean price per maker from TRAIN only
# insurer_price_cols = [c for c in df_train.columns if c.endswith('_price')]
# train_only = combined[combined['_is_train'] == 1]

# maker_encoding = train_only.groupby('vehicle_maker')[insurer_price_cols].mean().mean(axis=1)
# maker_encoding.name = 'vehicle_maker_encoded'

# combined = combined.merge(maker_encoding, on='vehicle_maker', how='left')
# combined.drop(columns=['vehicle_maker'], inplace=True)

# # ── Split back into train and test ───────────────────────────────────────
# df_train = combined[combined['_is_train'] == 1].drop(columns=['_is_train'])
# df_test  = combined[combined['_is_train'] == 0].drop(columns=['_is_train', 
#             *insurer_price_cols])

# df_train = df_train.copy()
# df_test  = df_test.copy()

# print(f"Train shape: {df_train.shape}")
# print(f"Test shape:  {df_test.shape}")
# print(f"Remaining object cols in train: {df_train.select_dtypes(include='object').columns.tolist()}")
# print(f"Remaining object cols in test:  {df_test.select_dtypes(include='object').columns.tolist()}")

Train shape: (541292, 101)
Test shape:  (164092, 90)
Remaining object cols in train: ['coverage']
Remaining object cols in test:  ['coverage']


In [16]:
insurer_price_cols = [c for c in df_train.columns if c.endswith('_price')]

# Mean price per color across all insurers
color_price = df_train.groupby('vehicle_primary_color')[insurer_price_cols].mean()
color_price['avg_price'] = color_price.mean(axis=1)
print(color_price['avg_price'].sort_values(ascending=False).round(2))

# Also check how much variance color explains
overall_mean = df_train[insurer_price_cols].mean().mean()
color_std = color_price['avg_price'].std()
print(f"\nOverall mean price: {overall_mean:.2f}")
print(f"Std across colors:  {color_std:.2f}")
print(f"Max difference:     {color_price['avg_price'].max() - color_price['avg_price'].min():.2f}")

KeyError: 'vehicle_primary_color'